# Phase 1 — Dữ liệu

Tải xLAM-60k, Glaive và các subset SmolTalk theo revision trong `configs/data.yaml`, chuyển về định dạng chung, loại mẫu quá `max_seq_len` (không cắt), rồi ghi `data/processed/*.jsonl` và `data/REPORT.md`. Sau đó tách seen/unseen theo nhóm chức năng và khử trùng lặp (`data/split/`, `data/SPLIT.md`). Dữ liệu được đẩy lên một dataset **private** trên HF Hub.

xLAM bị gated: tài khoản HF phải đồng ý điều khoản trên trang dataset trước. CPU runtime là đủ.


## 1. Cài thư viện

In [ ]:
%pip install -q transformers==5.20.2 huggingface-hub==1.32.0 psutil==7.2.2 safetensors==0.8.0 pyyaml==6.0.3 datasets==5.1.0
from importlib.metadata import version
wanted = {"transformers": "5.20.2", "huggingface-hub": "1.32.0", "psutil": "7.2.2", "safetensors": "0.8.0", "pyyaml": "6.0.3", "datasets": "5.1.0"}
installed = {name: version(name) for name in wanted}
print(installed)
assert installed == wanted, "Phiên bản chưa khớp requirements.txt; chạy lại cell này."


## 2. Nạp mã nguồn của dự án

In [ ]:
import base64, hashlib, json, os, sys
from pathlib import Path

payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == "c42f855922e7c149c741ef278fd103a038a5658c658c6f5cc5e82b0e9b641833"
root = Path("/content/sft-base-playground")
root.mkdir(parents=True, exist_ok=True)
for relative, source in json.loads(payload).items():
    target = root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
os.chdir(root)
sys.path.insert(0, str(root))
# Các object đã tạo (như lab) vẫn giữ nguyên; lần import tiếp dùng source mới.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
print("Source bundle SHA-256:", "c42f855922e7c149c741ef278fd103a038a5658c658c6f5cc5e82b0e9b641833")
print("Working directory:", root)


## 3. Kiểm tra mã

In [ ]:
import subprocess
checks = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], capture_output=True, text=True)
print(checks.stdout)
print(checks.stderr)
checks.check_returncode()


## 4. Đăng nhập HF Hub

Cần secret `HF_TOKEN` trong 🔑 **Secrets** của Colab, đã bật **Notebook access**. Chạy cell này **bằng nút ▶ trên giao diện Colab**: secret không đọc được khi cell được chạy qua MCP.

Điền `DATASET_REPO` dạng `<username>/<tên repo>`.

In [ ]:
from google.colab import userdata
from huggingface_hub import whoami

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
DATASET_REPO = ""  # ví dụ "your-username/sft-capability-data"
print("Logged in as:", whoami()["name"])
assert DATASET_REPO, "Điền DATASET_REPO trước khi chạy tiếp."


## 5. Tải, chuyển đổi, lọc độ dài và đẩy lên Hub
Khoảng 10–15 phút; log tiến độ hiện ngay trong lúc chạy.

In [ ]:
!python -m src.data.build --config configs/data.yaml --push-to {DATASET_REPO}


## 6. Báo cáo
Sao chép nội dung này vào `data/REPORT.md` trong repo.

In [ ]:
print(Path("data/REPORT.md").read_text(encoding="utf-8"))


## 7. Phân cụm tool, tách seen/unseen, khử trùng lặp

Embed mọi tool, phân thành nhóm chức năng, chọn khoảng 15% số cụm làm unseen, giữ lại các pool eval rồi bỏ khỏi train mọi mẫu trùng hoặc gần trùng với eval (PLAN 4.2, 4.5). Nếu runtime đã mất `data/processed`, dữ liệu được tải lại từ `DATASET_REPO`. Kết quả đẩy lên thư mục `split/` của dataset.

In [ ]:
!python -m src.data.split --config configs/data.yaml --from-hub {DATASET_REPO} --push-to {DATASET_REPO}


## 8. Báo cáo tách
Sao chép nội dung này vào `data/SPLIT.md` trong repo.

In [ ]:
print(Path("data/SPLIT.md").read_text(encoding="utf-8"))
